In [1]:
import pandas as pd
import numpy as np

In [2]:
# df = pd.read_csv(r'C:\Users\BOOLA\Desktop\Process_project\data\Equipments\Equipment Browser_v2.csv', sep= None)
# df_ = pd.read_csv(r'C:\Users\BOOLA\Desktop\Process_project\data\Equipments\Agbaou.csv', sep= None)
# df_check = pd.read_excel(r"C:\Users\BOOLA\Downloads\Classeur2.xlsx")

df_test_hour  = pd.read_csv(r"C:\Users\BOOLA\Desktop\Downtime\Agbaou\Processed\AGBAOU_DOWN_202508.csv", sep= None, encoding="latin-1")

C:\Users\BOOLA\AppData\Local\Temp\ipykernel_7712\1572514913.py:5: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support sep=None with delim_whitespace=False; you can avoid this warning by specifying engine='python'.
  df_test_hour  = pd.read_csv(r"C:\Users\BOOLA\Desktop\Downtime\Agbaou\Processed\AGBAOU_DOWN_202508.csv", sep= None, encoding="latin-1")


In [3]:
df_test_hour.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 987 entries, 0 to 986
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Minesite         987 non-null    object 
 1   YearMonth        987 non-null    object 
 2   Equip No         987 non-null    object 
 3   Model            987 non-null    object 
 4   Labour Type      987 non-null    object 
 5   Description CAT  987 non-null    object 
 6   WorkType         987 non-null    object 
 7   Comments         987 non-null    object 
 8   Start Hours      987 non-null    object 
 9   End Hours        987 non-null    object 
 10  DowntimeHours    987 non-null    float64
dtypes: float64(1), object(10)
memory usage: 84.9+ KB


In [20]:
import pandas as pd

def split_overlapping_intervals(
    df: pd.DataFrame,
    subset_cols: list[str],
    start_col: str = "Start Hours",
    end_col: str = "End Hours"
) -> pd.DataFrame:

    df = df.copy()

    df[start_col] = pd.to_datetime(df[start_col])
    df[end_col] = pd.to_datetime(df[end_col])

    group_cols = [c for c in subset_cols if c not in [start_col, end_col]]

    processed_rows = []
    keep_rows = []

    # 👉 on traite groupe par groupe
    for _, group in df.groupby(group_cols, sort=False):

        time_points = pd.unique(group[[start_col, end_col]].values.ravel())
        time_points.sort()

        starts = group[start_col].values
        ends = group[end_col].values

        # 👉 on marque les lignes impliquées dans des chevauchements
        overlap_mask = pd.Series(False, index=group.index)

        for i in range(len(time_points) - 1):
            seg_start = time_points[i]
            seg_end = time_points[i + 1]

            mask = (starts <= seg_start) & (ends >= seg_end)

            if mask.any():
                overlap_mask |= mask

        # =========================
        # 1. lignes SANS chevauchement
        # =========================
        keep_rows.append(group.loc[~overlap_mask])

        # =========================
        # 2. lignes AVEC chevauchement
        # =========================
        overlap_group = group.loc[overlap_mask]

        if not overlap_group.empty:

            starts_o = overlap_group[start_col].values
            ends_o = overlap_group[end_col].values

            for i in range(len(time_points) - 1):

                seg_start = time_points[i]
                seg_end = time_points[i + 1]

                mask = (starts_o <= seg_start) & (ends_o >= seg_end)

                if not mask.any():
                    continue

                active = overlap_group.loc[mask]

                # 👉 règle métier : priorité à la plus courte intervention
                durations = active[end_col] - active[start_col]
                chosen = active.iloc[durations.argmin()]

                new_row = chosen.to_dict()
                new_row[start_col] = seg_start
                new_row[end_col] = seg_end

                processed_rows.append(new_row)

    # =========================
    # CONCAT FINAL
    # =========================
    result = pd.concat(
        keep_rows + [pd.DataFrame(processed_rows)],
        ignore_index=True
    )

    return result

In [33]:
import pandas as pd

def split_overlapping_intervals(
    df: pd.DataFrame,
    subset_cols: list[str],
    start_col: str = "Start Hours",
    end_col: str = "End Hours"
) -> pd.DataFrame:

    df = df.copy()

    df[start_col] = pd.to_datetime(df[start_col])
    df[end_col] = pd.to_datetime(df[end_col])

    group_cols = [c for c in subset_cols if c not in [start_col, end_col]]

    processed_rows = []
    keep_groups = []

    # =========================
    # 1. traitement par groupe
    # =========================
    for _, group in df.groupby(group_cols, sort=False):

        time_points = pd.unique(group[[start_col, end_col]].values.ravel())
        time_points.sort()

        starts = group[start_col].values
        ends = group[end_col].values

        # =========================
        # 2. détecter chevauchement
        # =========================
        overlap_mask = pd.Series(False, index=group.index)

        for i in range(len(time_points) - 1):
            seg_start = time_points[i]
            seg_end = time_points[i + 1]

            mask = (starts < seg_end) & (ends > seg_start)
            overlap_mask |= pd.Series(mask, index=group.index)

        # =========================
        # 3. garder les lignes NON impactées
        # =========================
        keep_groups.append(group.loc[~overlap_mask])

        # =========================
        # 4. traiter uniquement les lignes impactées
        # =========================
        overlap_group = group.loc[overlap_mask]

        if not overlap_group.empty:

            starts_o = overlap_group[start_col].values
            ends_o = overlap_group[end_col].values

            for i in range(len(time_points) - 1):

                seg_start = time_points[i]
                seg_end = time_points[i + 1]

                mask = (starts_o < seg_end) & (ends_o > seg_start)

                if not mask.any():
                    continue

                active = overlap_group.loc[mask]

                # =========================
                # règle métier : choisir 1 ligne
                # =========================
                durations = active[end_col] - active[start_col]
                chosen = active.iloc[durations.argmin()]

                new_row = chosen.to_dict()
                new_row[start_col] = seg_start
                new_row[end_col] = seg_end

                processed_rows.append(new_row)

    # =========================
    # 5. concat final
    # =========================
    result = pd.concat(
        keep_groups + [pd.DataFrame(processed_rows)],
        ignore_index=True
    )

    # =========================
    # 6. sécurité anti-doublons
    # =========================
    result = result.drop_duplicates(
        subset=subset_cols + [start_col, end_col]
    )

    return result, pd.DataFrame(processed_rows)

In [34]:
df_new_, hold = split_overlapping_intervals(df_test_hour, subset_cols= ['Equip No'])

In [36]:
hold.sort_values(by= ['Equip No', 'Start Hours'])

,Minesite,YearMonth,Equip No,Model,Labour Type,Description CAT,WorkType,Comments,Start Hours,End Hours,DowntimeHours
296,Agbaou/Mota,2025-08,D109,D7R,ENGINE,Engine,Unplanned,Overheating,2025-08-03 07:00:00,2025-08-03 19:00:00,12.000000
297,Agbaou/Mota,2025-08,D109,D7R,INSPECTION,Daily Inspection,Planned,Daily inspection and grease,2025-08-04 07:05:00,2025-08-04 07:15:00,0.166667
298,Agbaou/Mota,2025-08,D109,D7R,INSPECTION,Daily Inspection,Planned,Daily Inspection and graese,2025-08-07 07:12:00,2025-08-07 07:19:00,0.116667
299,Agbaou/Mota,2025-08,D109,D7R,BLADE,GET,Unplanned,"blade and ripper can not hoist , solenoid harn...",2025-08-09 07:00:00,2025-08-09 17:06:00,10.100000
300,Agbaou/Mota,2025-08,D109,D7R,HYDRAULIC,"Hydraulic System (Pumps, hydraulic actuators, ...",Unplanned,"Hydraulic oil leak, replace two burst Hydrauli...",2025-08-10 07:00:00,2025-08-10 17:13:00,10.216667
...,...,...,...,...,...,...,...,...,...,...,...
444,Agbaou/Mota,2025-08,WD11,834H,INSPECTION,Daily Inspection,Planned,Daily inspection and grease,2025-08-18 13:33:00,2025-08-18 13:45:00,0.200000
445,Agbaou/Mota,2025-08,WD11,834H,INSPECTION,Daily Inspection,Planned,Daily inspection and grease,2025-08-19 07:15:00,2025-08-19 07:30:00,0.250000
446,Agbaou/Mota,2025-08,WD11,834H,INSPECTION,Daily Inspection,Planned,Daily inspection and grease,2025-08-20 07:29:00,2025-08-20 07:36:00,0.116667
447,Agbaou/Mota,2025-08,WD11,834H,INSPECTION,Daily Inspection,Planned,Daily inspection and grease,2025-08-21 07:49:00,2025-08-21 07:55:00,0.100000


In [45]:
df_new = df_new_.drop_duplicates(ignore_index= True)

In [64]:
df_new['diff'] = np.around((df_new['End Hours'] - df_new['Start Hours']), 2)
df_new['DowntimeHours'] = np.around(df_new['DowntimeHours'], 2)

C:\Users\BOOLA\AppData\Local\Temp\ipykernel_21292\3773230621.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new['diff'] = np.around((df_new['End Hours'] - df_new['Start Hours']), 2)
C:\Users\BOOLA\AppData\Local\Temp\ipykernel_21292\3773230621.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new['DowntimeHours'] = np.around(df_new['DowntimeHours'], 2)


In [65]:
df_new.sort_values(by= ['Equip No', 'Start Hours']).iloc[120:180, :]

,Minesite,YearMonth,Equip No,Model,Labour Type,Description CAT,WorkType,Comments,Start Hours,End Hours,DowntimeHours,diff
842,Agbaou/Mota,2025-08,D109,D7R,INSPECTION,Daily Inspection,Planned,Daily inspection and grease,2025-08-27 21:00:00,2025-08-27 21:17:00,0.28,0 days 00:17:00
879,Agbaou/Mota,2025-08,D109,D7R,ELECTRICAL,Electrical System,Unplanned,Battery issue ( Smoke ):repair of the headligh...,2025-08-28 19:30:00,2025-08-28 20:23:00,0.88,0 days 00:53:00
884,Agbaou/Mota,2025-08,D109,D7R,SERVICE,PM,Planned,PM SERVICE 250 Hrs,2025-08-28 21:30:00,2025-08-28 02:29:00,4.98,-1 days +04:59:00
941,Agbaou/Mota,2025-08,D109,D7R,ENGINE,Engine,Unplanned,Bad engine noise:job on going,2025-08-30 02:34:00,2025-08-30 07:00:00,4.43,0 days 04:26:00
967,Agbaou/Mota,2025-08,D109,D7R,INSPECTION,Daily Inspection,Planned,Daily inspection and grease,2025-08-31 07:00:00,2025-08-31 10:05:00,3.08,0 days 03:05:00
27,Agbaou/Mota,2025-08,D119,D9R,TRANSMISSION,Transmission,Unplanned,High transmission oil temp,2025-08-01 07:00:00,2025-08-01 07:00:00,24.00,0 days 00:00:00
59,Agbaou/Mota,2025-08,D119,D9R,TRANSMISSION,Transmission,Unplanned,High transmission oil temp,2025-08-02 07:00:00,2025-08-02 07:00:00,24.00,0 days 00:00:00
97,Agbaou/Mota,2025-08,D119,D9R,TRANSMISSION,Transmission,Unplanned,High transmission oil temp,2025-08-03 07:00:00,2025-08-03 07:00:00,24.00,0 days 00:00:00
135,Agbaou/Mota,2025-08,D119,D9R,TRANSMISSION,Transmission,Unplanned,High transmission oil temp,2025-08-04 07:00:00,2025-08-04 07:00:00,24.00,0 days 00:00:00
166,Agbaou/Mota,2025-08,D119,D9R,TRANSMISSION,Transmission,Unplanned,High transmission oil temp,2025-08-05 07:00:00,2025-08-05 07:00:00,24.00,0 days 00:00:00


In [29]:
minesite_model = 'Essakane'

In [30]:
df_browser_model = df[
            df.Minesite.fillna('')
            .str.strip().str.lower()
            .str.find(minesite_model.lower()) >= 0
        ].reset_index(drop= True)

In [31]:
df_browser_model['Equipment'] = df_browser_model['Equipment'] + 'ESS'

In [33]:
df_browser_model["Equip No"] = df_browser_model['On Site Id']

In [34]:
df_ = (
    df_browser_model
    .dropna(subset=["On Site Id"])
    .drop_duplicates(subset=["On Site Id"])
    .set_index("On Site Id")["Equipment"]
    .to_dict()
)

df_browser_model["Equip No"] = (
    df_browser_model["Equip No"]
    .map(df_)
    .fillna(df_browser_model["Equip No"])
)

Model.py

In [ ]:
import streamlit as st
import pandas as pd
import os
from st_aggrid import AgGrid, GridOptionsBuilder

from frontend.read_files import read_csv_file, read_excel_file
from frontend.dialogues import show_modal_downtime, show_modal_operating
from frontend.clean_state import init_state


# ==========================================================
# CONFIG
# ==========================================================
st.set_page_config(layout="wide")
st.title("📊 Data Processing Platform")


# ==========================================================
# SESSION STATE INITIALIZATION
# ==========================================================
def init_state_():
    default_states = {
        "df_model": None,
        "df_browser_model": None,
        "success_message_model": None,
        "file_name": None,
        "excel_file": None,
        "sheets": None,
        "selected_sheet": None,
        "minesite_model": None,
        "equipment_model": None,
        "df_missed": None,
        "choice": "Operating"
    }

    for key, value in default_states.items():
        if key not in st.session_state:
            st.session_state[key] = value


site_list = [
    'Agbaou/Mota', 'Bonikro/Mota', 'Essakane', 'Fekola',
    'Goulamina/CORICA', 'Kouroussa', 'Sangaredi/CBG',
    'Seguela', 'Siguiri', 'Simandou/Mota',
    'SNIM-Guelb', 'Tongon'
]


# ==========================================================
# FILE UPLOAD SECTION
# ==========================================================
uploaded_file = st.file_uploader(
    "Load Excel or CSV file",
    type=["xlsx", "csv", "xls", "xlsm"],
    width = 600
)

# if 'df_model' not in st.session_state:
init_state_()


# ================== RUN UPLOADING ============
if uploaded_file and uploaded_file.name != st.session_state.file_name:

    # RESET ENTIRELY AT EACH LOADING
    for key in [
       "df_model", "success_message_model", "file_name", "excel_file",
        "sheets", "selected_sheet", "minesite_model", "equipment_model",
        "df_missed", "df_browser_model"
    ]:
        st.session_state[key] = None
    st.session_state.choice = "Operating"

    init_state()

    file_name = uploaded_file.name.lower()

    # ================= CSV =================
    if file_name.endswith(".csv"):
        df_model = read_csv_file(uploaded_file)
        st.session_state.df_model = df_model.copy()

        st.session_state.success_message_model = "CSV file successfully loaded ✅"
        
        
    # ================= EXCEL =================
    else:
        try:
            excel_file, sheets = read_excel_file(uploaded_file)             
            st.session_state.excel_file = excel_file
            st.session_state.sheets = sheets

            st.session_state.success_message_model = "Excel file successfully loaded ✅"         
                              
        except Exception as e:
            st.error(f"Wrong file extension: {e}")
    
    st.session_state.file_name = uploaded_file.name


# ================= Display Success Message =================
if st.session_state.success_message_model is not None:    
    st.success(st.session_state.success_message_model)


# ==========================================================
# EXCEL SHEET
# ==========================================================
if st.session_state.sheets is not None:                

    left, right = st.columns(2)
    left.header(f"Sheets available = {len(st.session_state.sheets)}")

    selected_sheet = right.selectbox(
        "Choose a sheet",
        st.session_state.sheets,
        key= 'selected_sheet'
    )
    
    
    if selected_sheet:
        df_model = pd.read_excel(
            st.session_state.excel_file,
            sheet_name= selected_sheet
        )
        st.session_state.df_model = df_model.copy()
    

# ==========================================================
# DATA EDITING SECTION
# ==========================================================
if st.session_state.df_model is not None:

    df_model = st.session_state.df_model

    # ================= Mine Site =================
    minesite_model = st.selectbox("Select <site name>", site_list,
                                  key= 'minesite_model', width= 400,
                                  placeholder= 'Select mine site name')

    if minesite_model:

        BASE_DIR = os.path.dirname(
            os.path.dirname(os.path.abspath(__file__)))
        
        browser_path = os.path.join(
            BASE_DIR,
            "data",
            "Equipments",
            "Equipment Browser_v2.csv"
        )

        if not os.path.exists(browser_path):
            st.error(f"File not found: {browser_path}")
        else:
            df_browser_model = pd.read_csv(browser_path, sep=None, engine='python')

        st.session_state.df_browser_model = df_browser_model[
            df_browser_model.Minesite.fillna('')
            .str.strip().str.lower()
            .str.find(minesite_model.lower()) >= 0
        ].reset_index(drop= True)

        if "browser_path_model" not in st.session_state:
            st.session_state.browser_path_model = browser_path

        if "template_path_model" not in st.session_state:
            st.session_state.template_path_model = os.path.join(
                BASE_DIR,
                "data",
                "template",
                "Template.xlsx"
            )
        
        if "trainer_path_model" not in st.session_state:
            st.session_state.trainer_path_model = os.path.join(BASE_DIR, "data", "model")
            
        st.subheader("🗳️ Editable Table")

        # ================= AGGRID =================
        gb = GridOptionsBuilder.from_dataframe(df_model)
        gb.configure_default_column(
            editable=True,
            filter=True,
            sortable=True
        )
        gb.configure_selection("multiple")

        grid_options = gb.build()

        grid_response = AgGrid(
            df_model,
            gridOptions= grid_options,
            update_on="MODEL_CHANGED",
            fit_columns_on_grid_load= True,
            theme= "streamlit"
        )
 
        edited_df_model = pd.DataFrame(grid_response["data"])

        # Persist edited dataframe
        st.session_state.df_model = edited_df_model

        # ================= Equipment Column =================
        equipment_model = st.selectbox("Select <Equipment number column>",
                                 edited_df_model.columns, key= 'equipment_model',
                                 width= 500, placeholder= 'Select column that hold equipment identifier number')        
        
        if equipment_model:

            st.session_state.df_browser_model['Equip Label'] = st.session_state.df_browser_model['Equip Label'].astype(str)
            edited_df_model[equipment_model] = edited_df_model[equipment_model].astype(str)

            if minesite_model == 'Essakane':
                df_missed = st.session_state.df_browser_model[
                    ~st.session_state.df_browser_model['On Site Id'].isin(
                        edited_df_model[equipment_model].unique())][[
                            'On Site Id', "SerialNumber", 'Model', "Parent Product Family"
                        ]].reset_index(drop=True)
                df_missed.rename(columns= {'On Site Id': equipment_model}, inplace= True)

                if not df_missed.empty:
                    st.session_state.df_missed = df_missed

            else:
                df_missed = st.session_state.df_browser_model[
                    ~st.session_state.df_browser_model['Equip Label'].isin(
                        edited_df_model[equipment_model].unique())][[
                            'Equip Label', "SerialNumber", 'Model', "Parent Product Family"
                        ]].reset_index(drop=True)
                df_missed.rename(columns= {'Equip Label': equipment_model}, inplace= True)

                if not df_missed.empty:
                    st.session_state.df_missed = df_missed            
            
            # 🔹 Show if only df_missed existes
            if st.session_state.df_missed is not None:
                st.warning("⚠️ List of equipments not found in browser mapping")
                st.dataframe(st.session_state.df_missed)
            else:
                st.success('✅ All equipments are include data.')

            # ==================================================
            # PROCESSING SECTION
            # ==================================================
            st.subheader("✒️ PROCESS DATA")

            choice = st.radio(
                "Choose processing type:",
                ["Operating", "Downtime"],
                horizontal=True,
                key= 'choice'
            )
            
            if st.button("➡️ Run"):

                st.session_state.equipment = equipment_model
                st.session_state.minesite = minesite_model
                  
                              

                if st.session_state.choice == "Operating":
                    show_modal_operating()
                else:
                    show_modal_downtime()

 downtime processing

In [ ]:
# coding:utf-8
import streamlit as st
import pandas as pd
from st_aggrid import AgGrid, GridOptionsBuilder, GridUpdateMode

from backend.fetch_data import fetch_data, DatasetType
from backend.packages.filtering import format_yearmonth_column, BrowserMapping
from backend.errors_handling import errors_handling
from backend.predict.predictor import fill_description_cat
from frontend.clean_state import init_state


# =====================================================
# CONFIG
# =====================================================
st.set_page_config(page_title="Downtime Hours")
st.title("Downtime Data Processing")

ERROR_ORDER = [
    'missing_values', 'duplicates', 'start_hours_mismatch',
    'downtime_mismatch', 'Negative_Hours', 'Downtime_imbricated_period'
]



# =====================================================
# UTILS
# =====================================================
@st.cache_data
def convert_csv(df: pd.DataFrame) -> bytes:
    return df.to_csv(index=False, sep=";").encode("utf-8-sig")


def check_required_keys(keys: list):
    missing = [k for k in keys if k not in st.session_state]
    if missing:
        st.warning(f"Missing required keys: {missing}")
        st.stop()


def get_browser_df():
    df = st.session_state.get("df_browser_model")
    if df is None or (hasattr(df, "empty") and df.empty):
        st.warning("⚠️ No file selected.")
        st.stop()
    return df


def build_equipment_mapping(df_browser: pd.DataFrame) -> dict:
    minesite = st.session_state.get("minesite")
    column = "On Site Id" if minesite == "Essakane" else "Equip Label"
    return (
        df_browser
        .dropna(subset=[column])
        .drop_duplicates(subset=[column])
        .set_index(column)["Equipment"]
        .to_dict()
    )


def apply_equipment_mapping(df: pd.DataFrame, mapping: dict) -> pd.DataFrame:
    df = df.copy()
    equipment = st.session_state.get('equipment')
    df[equipment] = df[equipment].map(mapping).fillna(df[equipment])
    return df


def normalize_units(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    unit = st.session_state.get("time_unit_dt_hrs_dialog")
    if unit == "Min":
        df["DowntimeHours"] /= 60
    elif unit == "Sec":
        df["DowntimeHours"] /= 3600
    return df


def prepare_columns():
    mode = st.session_state.get("yearmonth_mode_dt_hrs_dialog")
    relevant = {
        "Equip No": st.session_state.get("equipment"),
        "Labour Type": st.session_state.get("labour_type_dialog"),
        "WorkType": st.session_state.get("work_type_dialog"),
        "Comments": st.session_state.get("comments_dialog"),
        "Start Hours": st.session_state.get("start_hours_dialog"),
        "End Hours": st.session_state.get("end_hours_dialog"),
        "DowntimeHours": st.session_state.get("downtime_hours_dialog")
    }

    if mode == "Select a column":
        relevant["YearMonth"] = st.session_state.get("year_month_column_dt_hrs_dialog")
        year_month = None
    else:
        year_month = {
            "YearMonth": st.session_state.get("year_month_value_dt_hrs_dialog")
        }

    return relevant, year_month


def to_string_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert all columns to string for safe UI editing.
    """
    return df.astype(str)


def restore_types(df: pd.DataFrame) -> pd.DataFrame:
    """Restore proper types after user editing (safe)."""
    df = df.copy()
    for col in ["Start Hours", "End Hours", "DowntimeHours"]:
        if col not in df.columns:
            continue
        if col in ["Start Hours", "End Hours"]:
            df[col] = pd.to_datetime(df[col], errors="coerce")
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df
 


# =====================================================
# INITIAL CHECKS
# =====================================================
check_required_keys([
    "labour_type_dialog", "work_type_dialog", "comments_dialog", "equipment",
    "start_hours_dialog", "end_hours_dialog", "yearmonth_mode_dt_hrs_dialog",
    "downtime_hours_dialog", "year_month_column_dt_hrs_dialog", "df_browser_model",
    "template_path_model"
])

df_browser = get_browser_df()
template_path = st.session_state.get("template_path_model")



# =====================================================
# PROCESSING
# =====================================================
if "df_model" in st.session_state:

    if "df_process" not in st.session_state:

        relevant_columns, year_month_mapping = prepare_columns()
        base_df = st.session_state.get("df_model").copy()

        if "equip_mapping" not in st.session_state:
            st.session_state.equip_mapping = build_equipment_mapping(df_browser)

        base_df = apply_equipment_mapping(base_df, st.session_state.equip_mapping)

        processed_df = fetch_data(
            dataset_type=DatasetType.DOWNTIME,
            dataset=base_df,
            template_path=template_path,
            df_browser=df_browser,
            selected_columns=relevant_columns,
            equip_column={"Equip No": "Equipment"},
            site={"Minesite": "Minesite"},
            year_month=year_month_mapping,
            mapping=BrowserMapping(
                df_key="Equip No",
                df_target="Model",
                browser_key="Equipment",
                browser_value="Model"
            ),
            numeric_columns="DowntimeHours",
            downtime=True
        )

        processed_df = normalize_units(processed_df)
        st.session_state.df_process = processed_df

    # ---- formatting datetime ----
    st.session_state.df_process = format_yearmonth_column(st.session_state.df_process)
    st.session_state.df_process["Start Hours"] = pd.to_datetime(
        st.session_state.df_process["Start Hours"], errors="coerce"
    ).dt.strftime("%Y-%m-%d %H:%M")
    st.session_state.df_process["End Hours"] = pd.to_datetime(
        st.session_state.df_process["End Hours"], errors="coerce"
    ).dt.strftime("%Y-%m-%d %H:%M")

    st.dataframe(st.session_state.df_process)



# =====================================================
# ERRORS HANDLING
# =====================================================
    df_process, errors_df = errors_handling(
        st.session_state.df_process,
        {"subset": ["YearMonth", "Equip No", "Labour Type", "WorkType", "Comments", "Start Hours"]},
        {"subset": ["Equip No", "Labour Type", "WorkType", "Start Hours", "End Hours", "DowntimeHours"]},
        outlier_columns=None,
        downtime=True
    )

    st.session_state.df_valid = restore_types(df_process)

    st.session_state.setdefault("error_step", 0)
    st.session_state.setdefault("current_error_index", 0)
    if "error_keys" not in st.session_state:
        st.session_state.error_keys = [e for e in ERROR_ORDER if e in errors_df]


# =====================================================
# WORKTYPE RENAME DIALOG
# =====================================================
@st.dialog("Rename values in <<WorkType>>")
def work_type_dialog():

    df = st.session_state.df_process.copy()

    column1 = [
        'Labour Type', 'WorkType', 'Comments'
    ]
    df[column1] = df[column1].apply(
        lambda col: col.str.replace(r"[\r\n]", "", regex=True)
    )

    column = "WorkType"
    unique_values = df[column].fillna("(vide)").unique()
    mapping = {}

    for val in unique_values:
        new_val = st.text_input(f"{val}", value=val, key=f"rename <<{column}>> --> {val}")
        mapping[val] = new_val

    if st.button("Apply renaming"):
        df[column] = df[column].map(mapping).fillna(df[column])
        st.session_state.df_process = df
        st.session_state.df_edited = True
        st.success("✅ Values renamed successfully")
        st.rerun()


# =====================================================
# UI FUNCTIONS
# =====================================================
def show_summary():

    st.warning("⚠️ Summary of dataset errors")
    summary = pd.DataFrame({
        "Error type": st.session_state.error_keys,
        "Rows count": [len(errors_df[k]) for k in st.session_state.error_keys]
    })
    st.dataframe(summary)

    if summary["Rows count"].eq(0).all():
        st.success("✅ Your dataset haven't got errors rows.")
        if st.button("Close"):
            st.session_state.df_process = fill_description_cat(st.session_state.df_process)
            st.session_state.error_step = 0
            st.rerun()
    else:
        st.error("🚨 There are few rows that hold errors.")

        if st.button("Next"):
            # --- Move to the first errors type no empty ---
            for i, k in enumerate(st.session_state.error_keys):
                if not errors_df[k].empty:
                    st.session_state.current_error_index = i
                    st.session_state.error_step = k
                    st.rerun()


# ====== Dialog managing each error available ======
def show_error_detail():

    # ---- Errors handling step by step -----
    key = st.session_state.error_step
    st.warning(f"⚠️ Handling {key.replace('_', ' ')}")
    
    # ------ Errors rows by type ----------
    grid_df = errors_df[key].copy()

    # --- Ensure to have  Start/End Hours in string type ---
    for col in ["Start Hours", "End Hours"]:
        if col in grid_df.columns:
            grid_df[col] = pd.to_datetime(grid_df[col], errors="coerce").dt.strftime("%Y-%m-%d %H:%M")
            grid_df[col] = grid_df[col].fillna("")

    grid_df["YearMonth"] = grid_df["YearMonth"].astype(str)

    # -------------------- AGGRID ---------------------
    gb = GridOptionsBuilder.from_dataframe(grid_df)
    gb.configure_default_column(editable=True, filter=True, sortable=True)

    # ------ FORCE to keep the both columns in string type ------------
    for col in ["Start Hours", "End Hours"]:
        gb.configure_column(
            col,
            editable=True,
            cellEditor="agTextCellEditor",
            type=["text"]
        )

    gb.configure_selection("multiple", use_checkbox=True)

    grid = AgGrid(
        grid_df,
        gridOptions=gb.build(),
        update_on="SELECTION_CHANGED",
        fit_columns_on_grid_load= True,
        theme= "streamlit",
        key=f"errors_grid_{key}"
    )

    # ------------- Recover selected rows ------------
    selected_rows = grid.get("selected_rows", [])
    if selected_rows is not None and len(selected_rows) > 0:
        selected = pd.DataFrame(selected_rows)
        
        # Drop columns internal added by AgGrid (_selectedRowNodeInfo, etc.)
        selected = selected.loc[:, ~selected.columns.str.startswith("_")]
        
        for col in ["Start Hours", "End Hours"]:
            if col in selected.columns:
                selected[col] = pd.to_datetime(selected[col], errors="coerce")
    else:
        selected = pd.DataFrame()

    # ----------- Buton Next or Confirm if the last -----------
    next_index = st.session_state.current_error_index + 1

    while next_index < len(st.session_state.error_keys) and errors_df[
        st.session_state.error_keys[next_index]].empty:

        next_index += 1

    if next_index < len(st.session_state.error_keys):
            if st.button("Next"):       
                st.session_state.df_process = pd.concat(
                    [st.session_state.df_valid, selected], ignore_index=True) if not selected.empty else st.session_state.df_valid
                
                st.session_state.current_error_index = next_index
                st.session_state.error_step = st.session_state.error_keys[next_index]
                st.rerun()
        
    else:
        if st.button("Confirm"):
            st.session_state.df_process = pd.concat(
                [st.session_state.df_valid, selected], ignore_index=True) if not selected.empty else st.session_state.df_valid
            
            st.session_state.df_process = fill_description_cat(st.session_state.df_process)
            st.session_state.error_step = 0
            st.session_state.df_edited = True
            st.rerun()


# =====================================================
# UI FLOW
# =====================================================
col_main, col_side = st.columns([9, 1])

with col_side:
    if st.button("Errors"):
        st.session_state.error_step = "summary"

if st.session_state.error_step == "summary":
    st.markdown("Deep view on rows that held errors...")
    show_summary()

elif st.session_state.error_step in st.session_state.error_keys:
    show_error_detail()

with col_main:
    st.session_state.setdefault("df_edited", False)
    
    if st.button("Rename", help= 'Use this button to rename *WorkType* column elements'):
        work_type_dialog()


# =====================================================
# DOWNLOAD
# =====================================================
if st.session_state.df_edited:
    file_name = st.text_input(
            "File name", width= 200,
            placeholder= "Enter file name")
    file_name = file_name.strip().replace(" ", "_")
    
    if file_name:
        st.download_button(
            label="⬇️ Download",
            data=convert_csv(st.session_state.df_process),
            file_name=f"{file_name}.csv",
            mime="text/csv"
        )


# =====================================================
# NAVIGATION
# =====================================================
if st.button("⬅️ Back"):
    keys_to_clear = [
        "df_process",
        "error_step",
        "error_keys",
        "current_error_index",
        "df_edited"
    ]

    for key in keys_to_clear:
        if key in st.session_state:
            st.session_state.pop(key, None)
            
    init_state()
    st.switch_page("model.py")

Operating processing

In [ ]:
# coding:utf-8
import streamlit as st
import pandas as pd
from st_aggrid import AgGrid, GridOptionsBuilder, GridUpdateMode

from backend.fetch_data import fetch_data, DatasetType
from backend.packages.filtering import format_yearmonth_column, BrowserMapping
from backend.errors_handling import errors_handling
from frontend.clean_state import init_state


# =====================================================
# CONFIG
# =====================================================
st.set_page_config(page_title="Operating Hours")
st.title("Operating Data Processing")

ERROR_ORDER = ['missing_values', 'outliers', 'duplicates', 'Negative_Hours']


# =====================================================
# UTILS
# =====================================================
@st.cache_data
def convert_csv(df: pd.DataFrame) -> bytes:
    return df.to_csv(index=False, sep=";").encode("utf-8-sig")


def check_required_keys(keys: list):
    missing = [k for k in keys if k not in st.session_state]
    if missing:
        st.warning(f"Missing required keys: {missing}")
        st.stop()


def get_browser_df():
    df = st.session_state.get("df_browser_model")
    if df is None or (hasattr(df, "empty") and df.empty):
        st.warning("⚠️ No file selected.")
        st.stop()
    return df


def build_equipment_mapping(df_browser: pd.DataFrame) -> dict:
    minesite = st.session_state.get("minesite")
    column = "On Site Id" if minesite == "Essakane" else "Equip Label"

    return (
        df_browser
        .dropna(subset=[column])
        .drop_duplicates(subset=[column])
        .set_index(column)["Equipment"]
        .to_dict()
    )


def apply_equipment_mapping(df: pd.DataFrame, mapping: dict) -> pd.DataFrame:
    df = df.copy()
    equipment = st.session_state.get('equipment')
    df[equipment] = df[equipment].map(mapping).fillna(df[equipment])
    return df


def normalize_units(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    unit = st.session_state.get("time_unit_op_hrs_dialog")

    if unit == "Min":
        df["SMU Hours"] /= 60
    elif unit == "Sec":
        df["SMU Hours"] /= 3600

    return df


def prepare_columns():
    mode = st.session_state.get("yearmonth_mode_op_hrs_dialog")

    relevant = {
        "Equipment": st.session_state.get("equipment"),
        "SMU Hours": st.session_state.get("smu_hours_dialog")
    }

    if mode == "Select a column":
        relevant["YearMonth"] = st.session_state.get(
            "year_month_column_op_hrs_dialog"
        )
        year_month = None
    else:
        year_month = {
            "YearMonth": st.session_state.get(
                "year_month_value_op_hrs_dialog"
            )
        }

    return relevant, year_month


# =====================================================
# INITIAL CHECKS
# =====================================================
check_required_keys([
    "equipment",
    "smu_hours_dialog",
    "yearmonth_mode_op_hrs_dialog"
])

df_browser = get_browser_df()
template_path = st.session_state.get("template_path_model")


# =====================================================
# PROCESSING
# =====================================================
if "df_model" in st.session_state:

    if "df_process" not in st.session_state:

        relevant_columns, year_month_mapping = prepare_columns()

        base_df = st.session_state.get("df_model")

        # ---- mapping ----
        if "equip_mapping" not in st.session_state:
            st.session_state.equip_mapping = build_equipment_mapping(df_browser)

        base_df = apply_equipment_mapping(
            base_df,
            st.session_state.equip_mapping
        )

        processed_df = fetch_data(
            dataset_type=DatasetType.OPERATING,
            dataset=base_df,
            template_path=template_path,
            df_browser=df_browser,
            selected_columns=relevant_columns,
            equip_column={"Equipment": "Equipment"},
            site={"Site": "Minesite"},
            year_month=year_month_mapping,
            mapping=BrowserMapping(
                df_key="Equipment",
                df_target="Model",
                browser_key="Equipment",
                browser_value="Model"
            ),
            numeric_columns="SMU Hours"
        )

        
        # ---- units ----
        processed_df = normalize_units(processed_df)

        st.session_state.df_process = processed_df

    # ---- formatting ----
    st.session_state.df_process = format_yearmonth_column(
        st.session_state.df_process
    )

    st.dataframe(st.session_state.df_process)


# =====================================================
# ERRORS HANDLING
# =====================================================
    df_process, errors_df = errors_handling(
        st.session_state.df_process,
        {"subset": ["Equipment", "YearMonth"]},
        {"subset": ["Equipment", "SMU Hours"]},
        {"column": "SMU Hours", "low": 0.1, "high": 730}
    )

    # ---- init state ----
    st.session_state.setdefault("error_step", 0)
    st.session_state.setdefault("current_error_index", 0)

    if "error_keys" not in st.session_state:
        st.session_state.error_keys = [
            e for e in ERROR_ORDER if e in errors_df
        ]

    # =================================================
    # UI FUNCTIONS
    # =================================================
    def show_summary():
        st.warning("⚠️ Summary of dataset errors")

        summary = pd.DataFrame({
            "Error type": st.session_state.error_keys,
            "Rows count": [
                len(errors_df[k]) for k in st.session_state.error_keys
            ]
        })

        st.dataframe(summary)

        if summary["Rows count"].sum() == 0:
            st.success("✅ No errors found.")
            if st.button("Close"):
                st.session_state.df_download = True
                st.session_state.error_step = 0
                st.rerun()
        else:
            if st.button("Next"):
                for i, k in enumerate(st.session_state.error_keys):
                    if not errors_df[k].empty:
                        st.session_state.current_error_index = i
                        st.session_state.error_step = k
                        st.rerun()

    def show_error_detail():
        key = st.session_state.error_step

        st.warning(f"⚠️ Handling {key.replace('_', ' ')}")

        gb = GridOptionsBuilder.from_dataframe(errors_df[key])
        gb.configure_default_column(editable=True, filter=True, sortable=True)
        gb.configure_selection("multiple", use_checkbox=True)

        grid = AgGrid(
            errors_df[key],
            gridOptions=gb.build(),
            update_mode=GridUpdateMode.SELECTION_CHANGED
        )

        selected = pd.DataFrame(grid.get("selected_rows", []))

        next_index = next(
            (i for i in range(
                st.session_state.current_error_index + 1,
                len(st.session_state.error_keys)
            ) if not errors_df[
                st.session_state.error_keys[i]
            ].empty),
            None
        )

        if next_index is not None:
            if st.button("Next"):
                st.session_state.df_process = pd.concat(
                    [df_process, selected], ignore_index=True
                )
                st.session_state.current_error_index = next_index
                st.session_state.error_step = (
                    st.session_state.error_keys[next_index]
                )
                st.rerun()
        else:
            if st.button("Confirm"):
                st.session_state.df_process = pd.concat(
                    [df_process, selected], ignore_index=True
                )
                st.session_state.df_download = True
                st.session_state.error_step = 0
                st.rerun()

    # =================================================
    # UI FLOW
    # =================================================
    col1, col2 = st.columns([9, 1])

    with col2:
        if st.button("Errors"):
            st.session_state.error_step = "summary"

    if st.session_state.error_step == "summary":
        show_summary()

    elif st.session_state.error_step in st.session_state.error_keys:
        show_error_detail()


# =====================================================
# DOWNLOAD
# =====================================================
    st.session_state.setdefault("df_download", False)

    if st.session_state.df_download:
        file_name = st.text_input("File Name")

        if file_name:
            st.download_button(
                label="⬇️ Download",
                data=convert_csv(st.session_state.df_process),
                file_name=f"{file_name}.csv",
                mime="text/csv"
            )


# =====================================================
# NAVIGATION
# =====================================================
if st.button("⬅️ Back"):
    for key in [
        "df_process",
        "error_step",
        "error_keys",
        "current_error_index",
        "df_download"
    ]:
        st.session_state.pop(key, None)

    init_state()
    st.switch_page("model.py")

In [ ]:
def smart_clean(df):
    df = df.replace(r'^\s*$', pd.NA, regex=True)

    for col in df.columns:
        # Si majorité numérique → convertir
        if pd.to_numeric(df[col], errors='coerce').notna().sum() > len(df) * 0.7:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df